# 01 — Bona Data Analysis
## SunnyBest Retail Forecasting System — Raw Database Tables

Analyses all source tables pulled directly from the Supabase PostgreSQL database (`core` schema).

**Dimension tables:** `dim_calendar`, `dim_policy_regimes`, `dim_products`, `dim_stores`

**Fact tables:** `fact_sales`, `fact_inventory`, `fact_customer_activity`, `fact_store_operations`, `fact_promotions`, `fact_restriction_events`, `fact_weather`

---
## 0. DB Connection & Load All Tables

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import warnings
from sqlalchemy import create_engine
from urllib.parse import quote_plus

warnings.filterwarnings("ignore")
plt.rcParams["figure.figsize"] = (14, 5)
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

# ------- Connection (fill in your credentials) -------
host     = "aws-1-eu-central-1.pooler.supabase.com"
port     = 5432
database = "postgres"
user     = "postgres.ogkdfmkybqtrsglcizzt"
password = quote_plus("YOUR_PASSWORD")   # replace with your password

engine = create_engine(
    f"postgresql+psycopg2://{user}:{password}@{host}:{port}/{database}",
    pool_pre_ping=True
)

print("Engine created.")

In [ ]:
# -------------------------
# DIM TABLES
# -------------------------
df_calendar = pd.read_sql("SELECT * FROM core.dim_calendar ORDER BY date ASC", engine)
df_policy_regimes = pd.read_sql("SELECT * FROM core.dim_policy_regimes ORDER BY start_date ASC", engine)
df_products = pd.read_sql("SELECT * FROM core.dim_products ORDER BY product_id ASC", engine)
df_stores = pd.read_sql("SELECT * FROM core.dim_stores ORDER BY store_id ASC", engine)

# -------------------------
# FACT TABLES
# -------------------------
df_sales = pd.read_sql("SELECT * FROM core.fact_sales ORDER BY date ASC", engine)
df_inventory = pd.read_sql("SELECT * FROM core.fact_inventory ORDER BY date ASC", engine)
df_customer_activity = pd.read_sql("SELECT * FROM core.fact_customer_activity ORDER BY date ASC", engine)
df_store_operations = pd.read_sql("SELECT * FROM core.fact_store_operations ORDER BY date ASC", engine)
df_promotions = pd.read_sql("SELECT * FROM core.fact_promotions ORDER BY date ASC", engine)
df_restrictions = pd.read_sql("SELECT * FROM core.fact_restriction_events ORDER BY date ASC", engine)
df_weather = pd.read_sql("SELECT * FROM core.fact_weather ORDER BY date ASC", engine)

print("All tables loaded successfully.")

---
## 1. Dataset Overview — All Tables at a Glance

In [ ]:
all_tables = {
    "dim_calendar":           df_calendar,
    "dim_policy_regimes":     df_policy_regimes,
    "dim_products":           df_products,
    "dim_stores":             df_stores,
    "fact_sales":             df_sales,
    "fact_inventory":         df_inventory,
    "fact_customer_activity": df_customer_activity,
    "fact_store_operations":  df_store_operations,
    "fact_promotions":        df_promotions,
    "fact_restriction_events":df_restrictions,
    "fact_weather":           df_weather,
}

rows = []
for name, d in all_tables.items():
    date_col = next((c for c in d.columns if "date" in c.lower()), None)
    date_range = f"{d[date_col].min()} → {d[date_col].max()}" if date_col else "—"
    missing_pct = round(d.isnull().sum().sum() / d.size * 100, 2)
    rows.append({
        "Table":       name,
        "Type":        "DIM" if name.startswith("dim") else "FACT",
        "Rows":        f"{len(d):,}",
        "Columns":     d.shape[1],
        "Missing %":   f"{missing_pct}%",
        "Date Range":  date_range,
    })

overview = pd.DataFrame(rows).set_index("Table")
display(overview)

In [ ]:
# Column-level missing values per table
for name, d in all_tables.items():
    missing = d.isnull().sum()
    missing = missing[missing > 0]
    if not missing.empty:
        print(f"\n{name} — columns with nulls:")
        for col, cnt in missing.items():
            print(f"  {col:<35} {cnt:>6} ({cnt/len(d)*100:.1f}%)")
    else:
        print(f"{name} — no missing values ✓")

---
## 2. Dimension Tables

### 2a. dim_stores

In [ ]:
print(f"Stores: {len(df_stores)}")
display(df_stores)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
df_stores["city"].value_counts().plot(kind="bar", ax=axes[0], color="#4C72B0", rot=30)
axes[0].set_title("Stores by City")

df_stores["store_size"].value_counts().plot(kind="bar", ax=axes[1], color="#55A868", rot=0)
axes[1].set_title("Stores by Size")

plt.tight_layout()
plt.show()

### 2b. dim_products

In [ ]:
print(f"Products: {len(df_products)}")
display(df_products.head(10))

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

df_products["category"].value_counts().sort_values().plot(kind="barh", ax=axes[0], color="#C44E52")
axes[0].set_title("Products by Category")

df_products["brand"].value_counts().head(10).sort_values().plot(kind="barh", ax=axes[1], color="#DD8452")
axes[1].set_title("Top 10 Brands")

plt.tight_layout()
plt.show()

print(f"\nPrice range: ₦{df_products['regular_price'].min():,.0f} — ₦{df_products['regular_price'].max():,.0f}")
print(f"Avg cost price: ₦{df_products['cost_price'].mean():,.0f}")